<a href="https://colab.research.google.com/github/Eunchae-L/Llama-Omni2-paper/blob/main/llama_omni2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Download

In [3]:
!hf download ICTNLP/InstructS2S-200K instruct_en_200k_data_cosy2.json --repo-type dataset --local-dir /content/instructs2s_small
!hf download ICTNLP/InstructS2S-200K en_part_00 --repo-type dataset --local-dir /content/instructs2s_small
!hf download ICTNLP/InstructS2S-200K en_part_01 --repo-type dataset --local-dir /content/instructs2s_small

!cat /content/instructs2s_small/en_part_00 /content/instructs2s_small/en_part_01 > /content/instructs2s_small/partial_instructs2s.tar.gz

!mkdir -p /content/sub_data
!tar -xzf /content/instructs2s_small/partial_instructs2s.tar.gz -C /content/sub_data

A new version of huggingface_hub (1.8.0) is available! You are using version 1.7.1.
To update, run: pip install -U huggingface_hub

instruct_en_200k_data_cosy2.json: 100% 1.06G/1.06G [00:04<00:00, 231MB/s]
/content/instructs2s_small/instruct_en_200k_data_cosy2.json
en_part_00: 100% 10.7G/10.7G [00:32<00:00, 329MB/s]
/content/instructs2s_small/en_part_00
en_part_01: 100% 10.7G/10.7G [00:32<00:00, 331MB/s]
/content/instructs2s_small/en_part_01

gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now


# Library Download

In [1]:
%cd /content

/content


In [2]:
!pip install -U transformer
!pip install -q openai-whisper

ERROR: Could not find a version that satisfies the requirement transformer (from versions: none)
ERROR: No matching distribution found for transformer
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 35.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 10.1 MB/s eta 0:00:00


# Library

In [4]:
import os
import re
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset

# Dataset

### Make aligned json

In [ ]:
# from collections import defaultdict
# from typing import Dict, List, Tuple, Optional


# def collect_existing_wavs(wav_root: str) -> Dict[str, Dict[int, Dict[str, str]]]:
#     """
#     실제 존재하는 wav 파일 스캔

#     반환 형식:
#     {
#         "instruct_en_100005": {
#             1: {
#                 "user": ".../instruct_en_100005-1-user.wav",
#                 "assistant": ".../instruct_en_100005-1-assistant.wav"
#             },
#             2: {...}
#         },
#         ...
#     }

#     user-prompt.wav 는 별도로 무시한다.
#     """
#     pattern = re.compile(r"^(instruct_en_\d+)-(\d+)-(user|assistant)\.wav$")
#     existing = defaultdict(lambda: defaultdict(dict))

#     for folder_name in os.listdir(wav_root):
#         folder_path = os.path.join(wav_root, folder_name)
#         if not os.path.isdir(folder_path):
#             continue

#         for fname in os.listdir(folder_path):
#             m = pattern.match(fname)
#             if m is None:
#                 continue

#             sample_id, turn_idx, speaker = m.group(1), int(m.group(2)), m.group(3)
#             full_path = os.path.join(folder_path, fname)
#             existing[sample_id][turn_idx][speaker] = full_path

#     return existing


# def speech_path_to_key(speech_path: str) -> Optional[Tuple[str, int, str]]:
#     """
#     json 안 speech 경로에서 sample_id, turn_idx, role(user/assistant) 를 추출한다.

#     예:
#     data/multiturn_instruction/en/wav/instruct_en_0/instruct_en_0-1-user.wav
#     -> ("instruct_en_0", 1, "user")
#     """
#     if speech_path is None:
#         return None

#     fname = os.path.basename(speech_path)
#     pattern = re.compile(r"^(instruct_en_\d+)-(\d+)-(user|assistant)\.wav$")
#     m = pattern.match(fname)
#     if m is None:
#         return None

#     sample_id, turn_idx, role = m.group(1), int(m.group(2)), m.group(3)
#     return sample_id, turn_idx, role


# def align_json_with_existing_wavs(
#     original_json_path: str,
#     wav_root: str,
#     output_json_path: str,
#     save_relative_path: bool = True,
# ):
#     """
#     실제 존재하는 wav 파일 기준으로 json을 정렬해 새 json을 만든다.

#     - 원본 json 로드
#     - wav_root 아래 실제 보유 wav 수집
#     - json의 각 turn 중 실제 wav가 존재하는 turn만 유지
#     - 결과 저장

#     save_relative_path=True 이면 output json의 speech 값은 wav_root 기준 상대경로로 저장
#     예: instruct_en_100005/instruct_en_100005-1-user.wav
#     """

#     with open(original_json_path, "r", encoding="utf-8") as f:
#         data = json.load(f)

#     existing = collect_existing_wavs(wav_root)

#     aligned_data = []
#     total_original_turns = 0
#     total_kept_turns = 0

#     for item in data:
#         sample_id = item.get("id")
#         conversation = item.get("conversation", [])
#         if not sample_id or not isinstance(conversation, list):
#             continue

#         if sample_id not in existing:
#             # 이 sample_id의 wav를 하나도 안 가지고 있으면 통째로 스킵
#             continue

#         new_conversation = []

#         for turn in conversation:
#             total_original_turns += 1

#             speech = turn.get("speech")
#             role = turn.get("from")

#             if speech is None:
#                 continue

#             parsed = speech_path_to_key(speech)
#             if parsed is None:
#                 continue

#             parsed_sample_id, turn_idx, speaker = parsed

#             # id mismatch면 스킵
#             if parsed_sample_id != sample_id:
#                 continue

#             # role과 speaker 정합성 확인
#             if role == "human" and speaker != "user":
#                 continue
#             if role == "gpt" and speaker != "assistant":
#                 continue

#             # 실제 wav 존재 여부 확인
#             if turn_idx not in existing[sample_id]:
#                 continue
#             if speaker not in existing[sample_id][turn_idx]:
#                 continue

#             real_wav_path = existing[sample_id][turn_idx][speaker]

#             new_turn = dict(turn)
#             if save_relative_path:
#                 new_turn["speech"] = os.path.relpath(real_wav_path, wav_root)
#             else:
#                 new_turn["speech"] = real_wav_path

#             new_conversation.append(new_turn)
#             total_kept_turns += 1

#         if len(new_conversation) == 0:
#             continue

#         aligned_item = {
#             "id": sample_id,
#             "conversation": new_conversation
#         }
#         aligned_data.append(aligned_item)

#     with open(output_json_path, "w", encoding="utf-8") as f:
#         json.dump(aligned_data, f, ensure_ascii=False, indent=2)

#     print(f"[Done] saved aligned json to: {output_json_path}")
#     print(f"original items: {len(data)}")
#     print(f"aligned items : {len(aligned_data)}")
#     print(f"original turns: {total_original_turns}")
#     print(f"kept turns    : {total_kept_turns}")

In [ ]:
# original_json_path = "/content/sub_data/en/instruct_en_200k_data_cosy1-25hz.json"
# wav_root = "/content/sub_data/en/wav"
# output_json_path = "/content/aligned_data.json"

# align_json_with_existing_wavs(
#     original_json_path=original_json_path,
#     wav_root=wav_root,
#     output_json_path=output_json_path,
#     save_relative_path=True,   # speech를 wav_root 기준 상대경로로 저장
# )

## Stage1(a) dataset

In [ ]:
class Stage1SpeechTextDataset(Dataset):
    def __init__(self, aligned_data, wav_root):
        self.wav_root = wav_root
        self.samples = []

        for item in aligned_data:
            sample_id = item["id"]
            conversation = item["conversation"]

            for i in range(len(conversation) - 1):
                current_turn = conversation[i]
                next_turn = conversation[i + 1]

                if current_turn["from"] != "human":
                    continue
                if next_turn["from"] != "gpt":
                    continue

                human_speech = current_turn["speech"]
                gpt_text = next_turn["text"]

                self.samples.append({
                    "id": sample_id,
                    "audio_path": os.path.join(self.wav_root, human_speech),
                    "target_text": gpt_text,
                    "input_text_optional": current_turn["text"],
                    "turn_index": i
                })

        if len(self.samples) == 0:
            raise ValueError("No samples found in the dataset.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        return sample

In [ ]:
aligned_json_path = "/content/aligned_data.json"
aligned_data = json.load(open(aligned_json_path, "r", encoding="utf-8"))

wav_root = "/content/sub_data/en/wav"

stage1a_dataset = Stage1SpeechTextDataset(aligned_data, wav_root)

In [ ]:
print("stage1(a):", len(stage1a_dataset))
print(stage1a_dataset[0])

stage1(a): 26249
{'id': 'instruct_en_21', 'audio_path': '/content/sub_data/en/wav/instruct_en_21/instruct_en_21-1-user.wav', 'target_text': 'The Cuban Missile Crisis was a thirteen-day standoff in nineteen sixty-two between the US and the Soviet Union over Soviet missiles in Cuba, which nearly led to nuclear war. The Vietnam War, on the other hand, was a prolonged conflict from nineteen fifty-five to nineteen seventy-five between the communist North Vietnam and the anti-communist South Vietnam, with the US providing military support to the South. While both events were Cold War conflicts, the Cuban Missile Crisis was a brief, high-stakes crisis averted by diplomacy, whereas the Vietnam War was a lengthy and bloody conflict with heavy US involvement.', 'input_text_optional': 'Hey, can you, like, compare and contrast the Cuban Missile Crisis and the Vietnam War for me?', 'turn_index': 0}


{

    'id': 'instruct_en_21',
    'audio_path': '/content/sub_data/en/wav/instruct_en_21/instruct_en_21-1-user.wav'
    'target_text': 'The Cuban Missile Crisis was ...,
    'input_text_optional': 'Hey, can you, like, compare and contrast the Cuban Missile Crisis and the Vietnam War for me?',
    'turn_index': 0
}


## Stage1(b) dataset

In [9]:
class Stage1BDataset(Dataset):
    """
    Stage I(b): <text response, speech response(unit)> pair dataset
    """

    def __init__(self, aligned_data):
        self.samples = []

        for item in aligned_data:
            conversation = item.get("conversation", [])
            sample_id = item.get("id", None)

            for turn_idx, turn in enumerate(conversation):
                if turn.get("from") != "gpt":
                    continue

                text = turn.get("text", "")
                unit_str = turn.get("unit", "")

                if not isinstance(text, str) or text.strip() == "":
                    continue
                if not isinstance(unit_str, str) or unit_str.strip() == "":
                    continue

                self.samples.append({
                    "id": sample_id,
                    "turn_index": turn_idx,
                    "text": text.strip(),
                    "unit_str": unit_str.strip(),
                })

        if len(self.samples) == 0:
            raise ValueError("Stage1BDataset: no valid samples found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [10]:
aligned_json_path = "/content/aligned_data.json"
aligned_data = json.load(open(aligned_json_path, "r", encoding="utf-8"))

wav_root = "/content/sub_data/en/wav"

stage1b_dataset = Stage1BDataset(aligned_data)

print(len(stage1b_dataset))

26251


## Stage2 dataset

In [2]:
# class Stage2Dataset(Dataset):
#     """
#     Stage2: human speech -> next gpt text + next gpt unit
#     """

#     def __init__(
#         self,
#         aligned_data: List[Dict[str, Any]],
#         wav_root: str,
#     ):
#         self.wav_root = wav_root
#         self.samples = []

#         for item in aligned_data:
#             sample_id = item["id"]
#             conv = item["conversation"]

#             for i in range(len(conv) - 1):
#                 cur_turn = conv[i]
#                 next_turn = conv[i + 1]

#                 if cur_turn.get("from") != "human":
#                     continue
#                 if next_turn.get("from") != "gpt":
#                     continue

#                 human_speech = cur_turn.get("speech")
#                 gpt_text = next_turn.get("text")
#                 gpt_unit = next_turn.get("unit")
#                 gpt_speech = next_turn.get("speech")

#                 if not human_speech or not gpt_text or not gpt_unit:
#                     continue

#                 self.samples.append({
#                     "id": sample_id,
#                     "audio_path": resolve_audio_path(wav_root, human_speech),
#                     "target_text": gpt_text,
#                     "target_unit": gpt_unit,
#                     "input_text_optional": cur_turn.get("text"),
#                     "assistant_audio_path_optional": resolve_audio_path(wav_root, gpt_speech) if gpt_speech else None,
#                     "turn_index": i,
#                 })

#         if len(self.samples) == 0:
#             raise ValueError("Stage2Dataset: no valid samples found.")

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):
#         return self.samples[idx]

# Speech Encoder

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class WhisperWrappedEncoder:
    @classmethod
    def load(cls):
        def replace_layer_norm(module):
            from whisper.model import LayerNorm
            for name, child in module.named_children():
                if isinstance(child, LayerNorm):
                    old_params = child.state_dict()
                    new_layer_norm = nn.LayerNorm(child.normalized_shape, eps=child.eps, elementwise_affine=child.elementwise_affine)
                    new_layer_norm.load_state_dict(old_params)
                    setattr(module, name, new_layer_norm)
                else:
                    replace_layer_norm(child)

        import whisper
        encoder = whisper.load_model(name="large-v3", device=device).encoder
        replace_layer_norm(encoder)
        return encoder

# Speech Adapter

In [ ]:
class EncoderProjectorConcat(nn.Module):
    def __init__(self):
        super().__init__()
        self.k = 5 #ds_rate
        self.encoder_dim = 1280 #whisper hiddensize
        self.llm_dim = 896 #llm hidden size
        self.linear1 = nn.Linear(self.encoder_dim * self.k, 2048)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(2048, self.llm_dim)

    def forward(self,x):
        batch_size, seq_len, dim = x.size() #(B, T, D)
        num_frames_to_discard = seq_len % self.k
        if num_frames_to_discard > 0:
            x = x[:, :-num_frames_to_discard, :] #남은 것은 버림
        seq_len = x.size(1)

        x = x.contiguous() #메모리 연속성 확보
        x = x.view(batch_size, seq_len // self.k, dim * self.k)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

# LLM

llava 기반 코드를 speech쪽으로 확장했다.

In [ ]:
import torch

DEFAULT_SPEECH_TOKEN = "<speech>"

def ensure_speech_token(tokenizer, model, speech_token=DEFAULT_SPEECH_TOKEN):
    """
    tokenizer에 <speech> special token을 추가하고,
    model embedding size도 tokenizer 길이에 맞게 늘린다.
    """
    old_vocab_size = len(tokenizer)

    # 이미 없으면 추가
    if speech_token not in tokenizer.get_vocab():
        num_added = tokenizer.add_special_tokens(
            {"additional_special_tokens": [speech_token]}
        )
    else:
        num_added = 0

    speech_token_id = tokenizer.convert_tokens_to_ids(speech_token)

    if speech_token_id is None or speech_token_id == tokenizer.unk_token_id:
        raise ValueError(f"{speech_token} token was not added correctly.")

    # tokenizer 길이가 바뀌었으면 모델 임베딩도 resize
    if len(tokenizer) != old_vocab_size:
        model.resize_token_embeddings(len(tokenizer))

        # 새로 추가된 토큰 임베딩을 기존 임베딩 평균으로 초기화
        with torch.no_grad():
            input_embeds = model.get_input_embeddings().weight
            output_embeds = model.get_output_embeddings().weight

            mean_input = input_embeds[:-num_added].mean(dim=0, keepdim=True)
            mean_output = output_embeds[:-num_added].mean(dim=0, keepdim=True)

            input_embeds[-num_added:] = mean_input
            output_embeds[-num_added:] = mean_output

    # config에도 저장
    model.config.speech_token = speech_token
    model.config.speech_token_id = speech_token_id

    return speech_token_id

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Optional, Tuple, Union

class Omni2SpeechMetaModel:
    """
    모델 내부에 붙는 speech 모듈을 보관하고 초기화
    """
    def __init__(self, config):
        self.config = config

        self.speech_encoder = WhisperWrappedEncoder.load()
        self.speech_projector = EncoderProjectorConcat()

    def get_speech_encoder(self):
        return getattr(self, "speech_encoder", None)

    def get_speech_projector(self):
        return getattr(self, "speech_projector", None)

class Omni2SpeechMetaForCausalLM(ABC):
    """
    speech를 받아서 LLM 입력으로 바꿔줌
    """
    @abstractmethod
    def get_model(self):
        pass

    def get_speech_encoder(self):
        return self.get_model().get_speech_encoder()

    def get_model_projector(self):
        return self.get_model().get_speech_projector()

    def encode_speech(self, speech: torch.FloatTensor, speech_lengths: torch.LongTensor):
        """
        Input:
        - speech: (B, T, 128) #(B,T,n_mels), large-v3는 n_mels=128
        - speech_lengths: (B,) #데이터가 실제로 존재하는 유효 길이
        """
        speech_encoder = self.get_speech_encoder()
        speech_projector = self.get_model_projector()

        #whisper encoder, 입력: (B, n_mels, T)(transpose)
        encoder_outs = speech_encoder(speech.permute(0, 2, 1))
        #whisper 통과 후 길이 절반 처리(whisper encoder가 2배 다운샘플링함)
        speech_lengths = (speech_lengths + 1) // 2

        #projector
        encoder_outs = speech_projector(encoder_outs)
        #projector의 k(ds rate)로 길이 축소
        speech_lengths = speech_lengths // speech_projector.k

        #batch별로 유효 길이만 잘라서 list로 반환
        speech_features = [
            encoder_outs[i, :speech_lengths[i]] for i in range(len(encoder_outs))
        ]
        return speech_features

    def prepare_inputs_labels_for_speech_and_text(
        self,
        input_ids: torch.LongTensor,
        position_ids: Optional[torch.LongTensor],
        attention_mask: Optional[torch.Tensor],
        past_key_values,
        labels: Optional[torch.LongTensor],
        speech: Optional[torch.FloatTensor],
        speech_lengths: Optional[torch.LongTensor],
    ):
        """
        input_ids 안의 SPEECH_TOKEN_INDEX 위치를 찾아서 해당 위치에 speech embedding sequence를 삽입한다
        """
        speech_encoder = self.get_speech_encoder()

        if speech_encoder is None or speech is None or input_ids.shape[1] == 1:
            return input_ids, position_ids, attention_mask, past_key_values, None, labels

        speech_features = self.encode_speech(speech, speech_lengths)

        _labels = labels #정답 토큰 id
        _position_ids = position_ids #각 토큰의 위치 인덱스
        _attention_mask = attention_mask #실제 입력과 padding 구분

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids, dtype=torch.bool)
        else:
            attention_mask = attention_mask.bool()

        if position_ids is None:
            position_ids = torch.arange(0, input_ids.shape[1], dtype=torch.long, device=input_ids.device)

        if labels is None:
            labels = torch.full_like(input_ids, IGNORE_INDEX)

        #입력된 text sequence에서 padding 제거
        input_ids = [
            cur_input_ids[cur_attention_mask] #boolean masking, True인 위치만 남기고 False인 위치는 버린다
            for cur_input_ids, cur_attention_mask in zip(input_ids, attention_mask)
        ]
        labels = [
            cur_labels[cur_attention_mask]
            for cur_labels, cur_attention_mask in zip(labels, attention_mask)
        ]

        new_input_embeds = []
        new_labels = []
        cur_speech_idx = 0

        for batch_idx, cur_input_ids in enumerate(input_ids):
            num_speech = (cur_input_ids == SPEECH_TOKEN_INDEX).sum()

            #speech token의 인덱스를 추출
            speech_token_indices = (
                [-1]
                + torch.where(cur_input_ids == SPEECH_TOKEN_INDEX)[0].tolist()
                + [cur_input_ids.shape[0]]
            ) #speech_token_indices: [-1, speechtoken-index1, ..., speechtoken-indexn, len(cur_input_ids)]

            cur_input_ids_nospeech = [] #speech가 아닌 텍스트 인덱스
            cur_labels = labels[batch_idx]
            cur_labels_nospeech = []

            #speech token 사이의 텍스트 구간을 잘라서 저장
            for i in range(len(speech_token_indices) - 1):
                start = speech_token_indices[i] + 1
                end = speech_token_indices[i + 1]
                cur_input_ids_nospeech.append(cur_input_ids[start:end]) #e.g.[[A],[B,C],[D]]
                cur_labels_nospeech.append(cur_labels[start:end])

            #텍스트 조각들의 길이 기록
            split_sizes = [x.shape[0] for x in cur_labels_nospeech] #e.g. [1,2,1]

            if sum(split_sizes) > 0: #텍스트가 하나라도 있다면 진행한다
                #torch.cat(...) -> [A,B,C,D]
                #토큰들을 임베딩으로 바꾼다
                text_embeds = self.get_model().embed_tokens(torch.cat(cur_input_ids_nospeech))
                cur_input_embeds_nospeech = torch.split(text_embeds, split_sizes, dim=0) #다시 조각크기대로 나눈다
            else:
                cur_input_embeds_nospeech = tuple(
                    torch.empty(
                        0,
                        self.get_model().config.hidden_size,
                        device=cur_input_ids.device,
                        dtype=self.get_model().embed_tokens.weight.dtype,
                    )
                    for _ in split_sizes
                )

            cur_new_input_embeds = [] #현재 샘플1개에 대한 최종 embedding 시퀀스
            cur_new_labels = [] #현재 샘플1개에 대한 최종 label 시퀀스

            for i in range(num_speech + 1):
                #text token은 speech token개수(num_speech)보다 1만큼 많음. 따라서 text token부터 넣고 시작
                cur_new_input_embeds.append(cur_input_embeds_nospeech[i])
                cur_new_labels.append(cur_labels_nospeech[i])

                if i < num_speech:
                    cur_speech_features = speech_features[cur_speech_idx]
                    cur_speech_idx += 1

                    cur_new_input_embeds.append(cur_speech_features)
                    cur_new_labels.append(
                        torch.full(
                            (cur_speech_features.shape[0],),
                            IGNORE_INDEX, #speech 구간 길이만큼 IGNORE_INDEX를 넣는다
                            device=cur_labels.device,
                            dtype=cur_labels.dtype,
                        )
                    )
            '''
            텍스트 임베딩 [A]
            speech 임베딩 [s1, s2, s3]
            텍스트 임베딩 [B,C]
            -> cat: [A, s1, s2, s3, B, C]
            '''
            cur_new_input_embeds = [x.to(self.device) for x in cur_new_input_embeds]
            cur_new_input_embeds = torch.cat(cur_new_input_embeds, dim=0)
            cur_new_labels = torch.cat(cur_new_labels, dim=0)

            #현재 샘플 결과를 배치 전체 리스트에 저장
            new_input_embeds.append(cur_new_input_embeds)
            new_labels.append(cur_new_labels)

        assert cur_speech_idx == len(speech_features) #모든 speech feature를 정확히 다 썼는지 확인

        # speech 삽입 후 길어질 수 있으므로 max length 자르기(모델에서 제시하는 max_length)
        tokenizer_model_max_length = getattr(self.config, "tokenizer_model_max_length", None)
        if tokenizer_model_max_length is not None:
            new_input_embeds = [x[:tokenizer_model_max_length] for x in new_input_embeds]
            new_labels = [x[:tokenizer_model_max_length] for x in new_labels]

        # max_len 길이에 맞춰서 패딩
        max_len = max(x.shape[0] for x in new_input_embeds) #현재 배치에서 가장 긴 시퀀스 길이
        batch_size = len(new_input_embeds) #샘플 개수

        new_input_embeds_padded = []
        new_labels_padded = torch.full(
            (batch_size, max_len),
            IGNORE_INDEX,
            dtype=new_labels[0].dtype,
            device=new_labels[0].device,
        ) #default: loss 계산 x
        attention_mask = torch.zeros(
            (batch_size, max_len),
            dtype=torch.bool,
            device=new_labels[0].device,
        ) #default: False(전부 padding)
        position_ids = torch.zeros(
            (batch_size, max_len),
            dtype=torch.long,
            device=new_labels[0].device,
        )

        #패딩방향, config의 tokenizer_padding_side가 없으면 기본값 "right"
        padding_side = getattr(self.config, "tokenizer_padding_side", "right")

        for i, (cur_new_embed, cur_new_labels) in enumerate(zip(new_input_embeds, new_labels)):
            cur_len = cur_new_embed.shape[0]

            if padding_side == "left":
                #부족한 길이만큼 0 embedding을 만들고 앞쪽에 붙인다
                pad_embed = torch.zeros(
                    (max_len - cur_len, cur_new_embed.shape[1]),
                    dtype=cur_new_embed.dtype,
                    device=cur_new_embed.device,
                )
                padded_embed = torch.cat([pad_embed, cur_new_embed], dim=0)
                new_input_embeds_padded.append(padded_embed)

                if cur_len > 0:
                    new_labels_padded[i, -cur_len:] = cur_new_labels
                    attention_mask[i, -cur_len:] = True
                    position_ids[i, -cur_len:] = torch.arange(
                        0, cur_len, dtype=torch.long, device=position_ids.device
                    )
            else:
                pad_embed = torch.zeros(
                    (max_len - cur_len, cur_new_embed.shape[1]),
                    dtype=cur_new_embed.dtype,
                    device=cur_new_embed.device,
                )
                padded_embed = torch.cat([cur_new_embed, pad_embed], dim=0)
                new_input_embeds_padded.append(padded_embed)

                if cur_len > 0:
                    new_labels_padded[i, :cur_len] = cur_new_labels
                    attention_mask[i, :cur_len] = True
                    position_ids[i, :cur_len] = torch.arange(
                        0, cur_len, dtype=torch.long, device=position_ids.device
                    )

        #샘플별 텐서 리스트 -> 3차원 텐서 (batch size, max len, hidden size)
        new_input_embeds = torch.stack(new_input_embeds_padded, dim=0)

        if _labels is None:
            new_labels = None
        else:
            new_labels = new_labels_padded

        if _attention_mask is None:
            attention_mask = None
        else:
            attention_mask = attention_mask.to(dtype=_attention_mask.dtype)

        if _position_ids is None:
            position_ids = None

        return None, position_ids, attention_mask, past_key_values, new_input_embeds, new_labels

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM, Qwen2Config, Qwen2Model, Qwen2ForCausalLM
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers.generation.utils import GenerateOutput


IGNORE_INDEX = -100
SPEECH_TOKEN_INDEX = -200

class Omni2SpeechQwen2Config(Qwen2Config):
    model_type = "omni2_speech_qwen2"


class Omni2SpeechQwen2Model(Omni2SpeechMetaModel, Qwen2Model):
    config_class = Omni2SpeechQwen2Config

    def __init__(self, config: Qwen2Config):
        Qwen2Model.__init__(self, config)
        Omni2SpeechMetaModel.__init__(self, config)


class Omni2SpeechQwen2ForCausalLM(Qwen2ForCausalLM, Omni2SpeechMetaForCausalLM):
    config_class = Omni2SpeechQwen2Config

    def __init__(self, config):
        # 공식 코드에서 Qwen2ForCausalLM의 기본 구성 대신 speech가 붙은 model/lm_head를 직접 구성함.
        super(Qwen2ForCausalLM, self).__init__(config)

        self.config = config
        self.model = Omni2SpeechQwen2Model(config)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # transformers pre-trained model init과 동일한 후처리
        self.post_init()

    def get_model(self):
        return self.model

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values=None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        speech: Optional[torch.FloatTensor] = None,
        speech_lengths: Optional[torch.LongTensor] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Tuple, CausalLMOutputWithPast]:

        if inputs_embeds is None:
            (
                input_ids,
                position_ids,
                attention_mask,
                past_key_values,
                inputs_embeds,
                labels,
            ) = self.prepare_inputs_labels_for_speech_and_text(
                input_ids=input_ids,
                position_ids=position_ids,
                attention_mask=attention_mask,
                past_key_values=past_key_values,
                labels=labels,
                speech=speech,
                speech_lengths=speech_lengths,
            )

        return Qwen2ForCausalLM.forward(
            self,
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            labels=labels,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            cache_position=cache_position,
        )

    @torch.no_grad()
    def generate(
        self,
        inputs: Optional[torch.Tensor] = None,
        speech: Optional[torch.Tensor] = None,
        speech_lengths: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> Union[GenerateOutput, torch.LongTensor]:

        position_ids = kwargs.pop("position_ids", None)
        attention_mask = kwargs.pop("attention_mask", None)

        if "inputs_embeds" in kwargs:
            raise NotImplementedError("inputs_embeds is not supported")

        if speech is not None:
            (
                inputs,
                position_ids,
                attention_mask,
                _,
                inputs_embeds,
                _,
            ) = self.prepare_inputs_labels_for_speech_and_text(
                input_ids=inputs,
                position_ids=position_ids,
                attention_mask=attention_mask,
                past_key_values=None,
                labels=None,
                speech=speech,
                speech_lengths=speech_lengths,
            )
        else:
            inputs_embeds = self.get_model().embed_tokens(inputs)

        return Qwen2ForCausalLM.generate(
            self,
            position_ids=position_ids,
            attention_mask=attention_mask,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )

    def prepare_inputs_for_generation(
        self,
        input_ids,
        past_key_values=None,
        inputs_embeds=None,
        **kwargs
    ):
        speech = kwargs.pop("speech", None)
        speech_lengths = kwargs.pop("speech_lengths", None)

        inputs = Qwen2ForCausalLM.prepare_inputs_for_generation(
            self,
            input_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )

        if speech is not None:
            inputs["speech"] = speech
            inputs["speech_lengths"] = speech_lengths

        return inputs


AutoConfig.register("omni2_speech_qwen2", Omni2SpeechQwen2Config)
AutoModelForCausalLM.register(Omni2SpeechQwen2Config, Omni2SpeechQwen2ForCausalLM)

In [ ]:
# import sys

# if "__main__" in sys.modules and not hasattr(sys.modules["__main__"], "__file__"):
#     sys.modules["__main__"].__file__ = "/content/notebook_runtime.py"

In [ ]:
from transformers import AutoTokenizer, AutoConfig

base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_cfg = AutoConfig.from_pretrained(base_model_name)
config = Omni2SpeechQwen2Config(**base_cfg.to_dict())
config.tokenizer_padding_side = "right"
config.tokenizer_model_max_length = 2048

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

model = Omni2SpeechQwen2ForCausalLM.from_pretrained(
    base_model_name,
    config=config,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
)

# <speech> 토큰 추가가 아직 안 되어 있으면 여기서 추가
speech_token_id = ensure_speech_token(tokenizer, model)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|█████████████████████████████████████| 2.88G/2.88G [00:35<00:00, 86.0MiB/s]


# TTS

In [11]:
import ast
from types import SimpleNamespace
from typing import Optional, Tuple, Union

import torch
import torch.nn as nn
from transformers import Qwen2Config, Qwen2ForCausalLM

#각 샘플 길이로부터 attention mask 생성
def lengths_to_attention_mask(lens):
    bsz, max_lens = lens.size(0), torch.max(lens).item()
    mask = torch.arange(max_lens).to(lens.device).view(1, max_lens)
    mask = mask.expand(bsz, -1) >= lens.view(bsz, 1).expand(-1, max_lens)
    return ~mask

#LLM hidden state를 TTS LM hidden size로 변환
class HiddenStateProjector(nn.Module):
    """
    hidden_size -> hidden_size*2 -> tts_hidden_size
    """
    def __init__(self, hidden_size, tts_hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.ReLU(),
            nn.Linear(hidden_size * 2, tts_hidden_size),
        )
    def forward(self, x):
        return self.net(x)

class GateFusion(nn.Module):
    """
    g = sigmoid(W [rep || emb] + b)
    c = g * rep + (1-g) * emb
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(2 * hidden_size, hidden_size),
            nn.Sigmoid(),
        )
    def forward(self, rep, emb):
        gate = self.gate(torch.cat([rep, emb], dim=-1))
        return rep * gate + emb * (1 - gate)

In [12]:
class LLMSpeechGenerator(nn.Module):
    """
    TTS model

    Stage I(b):
        - gate fusion bypass
        - text embedding만 사용

    Stage II:
        - projected LLM hidden + text embedding을 gate fusion
        - MTTS + gate/input_proj 학습
    """
    def __init__(self, config, tts_tokenizer):
        super().__init__()

        if not hasattr(config, "speech_generator"):
            raise ValueError("config.speech_generator is required")
        sg_cfg = config.speech_generator #Omni2의 TTS 관련 config
        if isinstance(sg_cfg, Qwen2Config):
            tts_cfg = sg_cfg
        elif isinstance(sg_cfg, dict):
            tts_cfg = Qwen2Config(**sg_cfg)
        else:
            tts_cfg = Qwen2Config(**vars(sg_cfg))

        self.config = config
        self.tts_tokenizer = tts_tokenizer
        self.model = Qwen2ForCausalLM(tts_cfg)

        # LLM hidden -> TTS hidden
        self.input_proj = HiddenStateProjector(
            config.hidden_size, #llm hidden size
            self.model.config.hidden_size, #tts hidden size
        )

        self.gate_fusion = GateFusion(self.model.config.hidden_size)

        self.stream_r, self.stream_w = self._parse_stream_params(
            getattr(config, "stream_params", (1,1))
        )
        # TTS 입력 시퀀스 최대 길이 제한, 4096
        self.max_seq_len = getattr(config, "tokenizer_model_max_length")

        # 생성이 끝났음을 알려주는 종료용 special token id <sep> (공식코드)
        self.sep_token_id = self._resolve_sep_token_id()

    @staticmethod
    def _parse_stream_params(stream_params):
        if isinstance(stream_params, str):
            stream_params = ast.literal_eval(stream_params)

        if not (isinstance(stream_params, (tuple, list)) and len(stream_params) == 2):
            raise ValueError(f"stream_params must be a pair, got {stream_params}")

        r, w = int(stream_params[0]), int(stream_params[1])
        if r <= 0 or w <= 0:
            raise ValueError(f"stream_params must be positive, got {(r, w)}")
        return r, w

    def _resolve_sep_token_id(self) -> Optional[int]:
        # 1) official convention: "<sep>"
        if hasattr(self.tts_tokenizer, "convert_tokens_to_ids"):
            sep_id = self.tts_tokenizer.convert_tokens_to_ids("<sep>")
            unk_id = getattr(self.tts_tokenizer, "unk_token_id", None)
            if sep_id is not None and sep_id != unk_id:
                return int(sep_id)

        # 2) fallback
        if getattr(self.tts_tokenizer, "sep_token_id", None) is not None:
            return int(self.tts_tokenizer.sep_token_id)
        if getattr(self.tts_tokenizer, "eos_token_id", None) is not None:
            return int(self.tts_tokenizer.eos_token_id)

        return None

    # TTS 모델 내부의 embedding layer 반환
    def get_input_embeddings(self):
        return self.model.get_input_embeddings()

    def set_training_stage(self, stage: str) -> None:
        """
        stage1b: TTS만 학습, gate/input_proj freeze
        stage2 : TTS + gate/input_proj 학습
        """
        stage = stage.lower()

        for p in self.model.parameters():
            p.requires_grad = True

        gate_trainable = (stage == "stage2")
        for p in self.input_proj.parameters():
            p.requires_grad = gate_trainable
        for p in self.gate_fusion.parameters():
            p.requires_grad = gate_trainable

    def embed_text(self, text_token_ids: torch.LongTensor) -> torch.Tensor:
        return self.get_input_embeddings()(text_token_ids)

    def project_hidden(self, llm_hidden_states: torch.Tensor) -> torch.Tensor:
        return self.input_proj(llm_hidden_states)

    def build_conditioning(
        self,
        text_token_ids,
        llm_hidden_states= None,
        fusion_mode= "gate",
    ):
        """
        fusion_mode:
            - 'text_only': Stage I(b), gate bypass
            - 'gate'     : Stage II / inference
        """
        fusion_mode = fusion_mode.lower()
        text_embeds = self.embed_text(text_token_ids)

        if fusion_mode == "text_only":
            return text_embeds

        if fusion_mode != "gate":
            raise ValueError(f"Unknown fusion_mode: {fusion_mode}")

        if llm_hidden_states is None:
            raise ValueError("llm_hidden_states is required when fusion_mode='gate'")

        if llm_hidden_states.shape[:2] != text_token_ids.shape:
            raise ValueError(
                f"llm_hidden_states and text_token_ids must align on (B,T), "
                f"got {tuple(llm_hidden_states.shape[:2])} vs {tuple(text_token_ids.shape)}"
            )

        projected = self.project_hidden(llm_hidden_states)
        return self.gate_fusion(projected, text_embeds)


    def _pack_single_stream(
        self,
        cond_embeds: torch.Tensor,       # (N, D)
        unit_token_ids: torch.LongTensor,  # (M,)
        append_sep: bool,
    ):
        """
        TTS 학습용 입력 시퀀스를 한 샘플 기준으로 조립
        teacher forcing용 fused conditiong C와 speech unit Y^U를 R:W 규칙으로 섞는다.

        입력:
         - cond_embeds: (N, D)
            : conditioning embeddings
            : Stage1(b)는 text embedding만, Stage2 or inference는 gate fusion된 embedding
         - unit_token_ids: (M,)

        출력:
        [C chunk][U chunk][C chunk][U chunk]...

        labels는 speech unit 위치에만 넣고, conditioning 위치는 IGNORE_INDEX로 둔다.
        """
        embed_layer = self.get_input_embeddings()

        seq_embeds = []
        seq_labels = []

        c_pos = 0
        u_pos = 0
        n_cond = cond_embeds.size(0)
        n_unit = unit_token_ids.size(0)

        while c_pos < n_cond or u_pos < n_unit:
            #conditioning chunk
            if c_pos < n_cond:
                take_c = min(self.stream_r, n_cond - c_pos) #R개까지, R개보다 적게 남았으면 남은 것만 가져옴
                seq_embeds.append(cond_embeds[c_pos:c_pos + take_c])
                seq_labels.append(
                    torch.full(
                        (take_c,),
                        IGNORE_INDEX,
                        dtype=torch.long,
                        device=unit_token_ids.device,
                    )
                ) #speech unit 예측이 목표, conditioning 위치는 정답을 맞힐 필요가 없음. 따라서 loss계산에서 무시되도록
                c_pos += take_c

            #speech unit chunk
            if u_pos < n_unit:
                take_u = min(self.stream_w, n_unit - u_pos)
                cur_units = unit_token_ids[u_pos:u_pos + take_u] #현재 token id
                seq_embeds.append(embed_layer(cur_units)) #embedding vector로 바꿔서 seq_embeds에 넣는다
                seq_labels.append(cur_units)
                u_pos += take_u

        # speech unit 생성이 끝난 뒤 <sep>까지 예측하도록 학습
        if append_sep and self.sep_token_id is not None:
            sep = unit_token_ids.new_tensor([self.sep_token_id])
            seq_embeds.append(embed_layer(sep))
            seq_labels.append(sep)

        packed_embeds = torch.cat(seq_embeds, dim=0)
        packed_labels = torch.cat(seq_labels, dim=0)

        if self.max_seq_len is not None and packed_embeds.size(0) > self.max_seq_len:
            packed_embeds = packed_embeds[:self.max_seq_len] #모델에 넣을 입력 embedding 시퀀스
            packed_labels = packed_labels[:self.max_seq_len] #loss 계산용 label 시퀀스

        return packed_embeds, packed_labels

    # 배치 단위로 스트리밍 입력을 만든다.
    def build_streaming_batch(
        self,
        conditioning_embeds: torch.Tensor,      # (B, N, D)
        text_attention_mask: Optional[torch.Tensor],
        unit_token_ids: torch.LongTensor,       # (B, M)
        unit_attention_mask: Optional[torch.Tensor],
        append_sep: bool = True,
    ):
        """
        return:
            inputs_embeds : (B, L, D)
            attention_mask: (B, L)
            labels        : (B, L)
            packed_lengths: (B,)
        """
        bsz, _, dim = conditioning_embeds.shape
        device = conditioning_embeds.device

        #각 샘플 별 conditioning 토큰 개수
        if text_attention_mask is None:
            text_lens = torch.full(
                (bsz,),
                conditioning_embeds.size(1),
                dtype=torch.long,
                device=device,
            )
        else:
            text_lens = text_attention_mask.long().sum(dim=-1)

        #각 샘플 별 unit 토큰 개수
        if unit_attention_mask is None:
            unit_lens = torch.full(
                (bsz,),
                unit_token_ids.size(1),
                dtype=torch.long,
                device=unit_token_ids.device,
            )
        else:
            unit_lens = unit_attention_mask.long().sum(dim=-1)

        packed_embeds_list = []
        packed_labels_list = []
        packed_lengths = []

        for i in range(bsz):
            cur_cond = conditioning_embeds[i, :text_lens[i]]
            cur_units = unit_token_ids[i, :unit_lens[i]]

            #한 샘플의 최종 입력 embedding, 한 샘플의 정답 label
            cur_embeds, cur_labels = self._pack_single_stream(
                cond_embeds=cur_cond,
                unit_token_ids=cur_units,
                append_sep=append_sep,
            ) #[C chunk][U chunk][C chunk][U chunk]... / [IGN IGN] [U1 U2 U3] [IGN IGN]...

            packed_embeds_list.append(cur_embeds)
            packed_labels_list.append(cur_labels)
            packed_lengths.append(cur_embeds.size(0))

        #배치 내 샘플 길이가 다르므로 가장 긴 길이에 맞춰서 나머지 패딩
        packed_lengths = torch.tensor(packed_lengths, dtype=torch.long, device=device)
        max_len = int(packed_lengths.max().item()) if bsz > 0 else 0

        batch_embeds = conditioning_embeds.new_zeros((bsz, max_len, dim)) #(B, max_len, D)
        batch_labels = torch.full(
            (bsz, max_len),
            IGNORE_INDEX,
            dtype=torch.long,
            device=unit_token_ids.device,
        ) #(B, max_len)

        #패딩된 샘플들 왼쪽 정렬, 즉 오른쪽 패딩
        for i, (cur_embeds, cur_labels) in enumerate(zip(packed_embeds_list, packed_labels_list)):
            cur_len = cur_embeds.size(0)
            batch_embeds[i, :cur_len] = cur_embeds
            batch_labels[i, :cur_len] = cur_labels

        #실제 토큰 위치: True, 패딩 위치: False
        attention_mask = lengths_to_attention_mask(packed_lengths)
        return batch_embeds, attention_mask, batch_labels, packed_lengths

    def forward(
        self,
        text_token_ids,            # (B, N), TTS tokenizer ids
        unit_token_ids,            # (B, M), TTS tokenizer ids
        llm_hidden_states= None,  # (B, N, main_llm_hidden)
        text_attention_mask= None,
        unit_attention_mask= None,
        fusion_mode: str = "gate",
        append_sep: bool = True,
        return_dict: bool = True,
    ):
        conditioning_embeds = self.build_conditioning(
            text_token_ids=text_token_ids,
            llm_hidden_states=llm_hidden_states,
            fusion_mode=fusion_mode,
        )#최종 conditioning embedding을 만든다(text only or gate fusion)

        inputs_embeds, attention_mask, labels, _ = self.build_streaming_batch(
            conditioning_embeds=conditioning_embeds,
            text_attention_mask=text_attention_mask,
            unit_token_ids=unit_token_ids,
            unit_attention_mask=unit_attention_mask,
            append_sep=append_sep,
        )#배치 전체를 [C][U][C][U] 형식으로 조립

        return self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=return_dict,
        )

    def forward_stage1b(
        self,
        text_token_ids: torch.LongTensor,
        unit_token_ids: torch.LongTensor,
        text_attention_mask: Optional[torch.Tensor] = None,
        unit_attention_mask: Optional[torch.Tensor] = None,
        append_sep: bool = True,
        return_dict: bool = True,
    ):
        return self.forward(
            text_token_ids=text_token_ids,
            unit_token_ids=unit_token_ids,
            llm_hidden_states=None,
            text_attention_mask=text_attention_mask,
            unit_attention_mask=unit_attention_mask,
            fusion_mode="text_only",
            append_sep=append_sep,
            return_dict=return_dict,
        )

    def forward_stage2(
        self,
        text_token_ids: torch.LongTensor,
        unit_token_ids: torch.LongTensor,
        llm_hidden_states: torch.Tensor,
        text_attention_mask: Optional[torch.Tensor] = None,
        unit_attention_mask: Optional[torch.Tensor] = None,
        append_sep: bool = True,
        return_dict: bool = True,
    ):
        return self.forward(
            text_token_ids=text_token_ids,
            unit_token_ids=unit_token_ids,
            llm_hidden_states=llm_hidden_states,
            text_attention_mask=text_attention_mask,
            unit_attention_mask=unit_attention_mask,
            fusion_mode="gate",
            append_sep=append_sep,
            return_dict=return_dict,
        )

    @torch.no_grad()
    def generate_units(
        self,
        tts_inputs: Optional[torch.Tensor],      # (T_prev, D) or None
        new_hidden_states: torch.Tensor,         # (T_new, main_llm_hidden)
        new_text_token_ids: torch.LongTensor,    # (T_new,), TTS tokenizer ids
        is_finished: bool = False,
        fusion_mode: str = "gate",
        temperature: float = 1.0,
        top_p: float = 1.0,
        do_sample: bool = True,
    ):
        """
        공개 speech_generator.py의 incremental generation 로직을 거의 그대로 유지.
        batch size 1 전제.
        """
        if new_hidden_states.ndim != 2:
            raise ValueError("new_hidden_states must have shape (T_new, hidden_size)")
        if new_text_token_ids.ndim != 1:
            raise ValueError("new_text_token_ids must have shape (T_new,)")

        if fusion_mode == "text_only":
            new_condition = self.embed_text(new_text_token_ids.unsqueeze(0)).squeeze(0)
        else:
            projected = self.project_hidden(new_hidden_states.unsqueeze(0)).squeeze(0)
            text_embeds = self.embed_text(new_text_token_ids.unsqueeze(0)).squeeze(0)
            new_condition = self.gate_fusion(projected, text_embeds)

        if tts_inputs is not None:
            tts_inputs = torch.cat([tts_inputs, new_condition], dim=0)
        else:
            tts_inputs = new_condition

        if is_finished and self.sep_token_id is not None:
            sep = new_text_token_ids.new_tensor([self.sep_token_id])
            sep_embed = self.get_input_embeddings()(sep)
            tts_inputs = torch.cat([tts_inputs, sep_embed], dim=0)

        max_new_tokens = self.stream_w if not is_finished else 1024

        outputs = self.model.generate(
            inputs_embeds=tts_inputs.unsqueeze(0),
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            num_beams=1,
            max_new_tokens=max_new_tokens,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        generated_tokens = outputs[0]
        generated_units = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

        generated_token_embeds = self.get_input_embeddings()(generated_tokens)
        tts_inputs = torch.cat([tts_inputs, generated_token_embeds], dim=0)

        return tts_inputs, generated_units

In [13]:
from types import SimpleNamespace

#Qwen2 기반 TTS 설정
def build_example_tts_config():
    return SimpleNamespace(
        hidden_size=896,
        tokenizer_model_max_length=4096,
        stream_params=(3, 10),
        unit_vocab_size=6561,
        speech_generator={
            "architectures": ["Qwen2ForCausalLM"],
            "attention_dropout": 0.0,
            "bos_token_id": 151643,
            "eos_token_id": 151643,
            "hidden_act": "silu",
            "hidden_size": 896,
            "initializer_range": 0.02,
            "intermediate_size": 4864,
            "max_position_embeddings": 32768,
            "max_window_layers": 24,
            "model_type": "qwen2",
            "num_attention_heads": 14,
            "num_hidden_layers": 24,
            "num_key_value_heads": 2,
            "rms_norm_eps": 1e-6,
            "rope_theta": 1000000.0,
            "sliding_window": None,
            "tie_word_embeddings": True,
            "torch_dtype": "bfloat16",
            "transformers_version": "4.43.4",
            "use_cache": True,
            "use_mrope": False,
            "use_sliding_window": False,
            "vocab_size": 158227,
        },
    )

# Flow Matching

In [ ]:
# 0) repo 설치/준비
!git clone https://github.com/ictnlp/LLaMA-Omni2.git
%cd LLaMA-Omni2
!pip install -e .

# 1) Cosy2 decoder 다운로드
!huggingface-cli download --resume-download ICTNLP/cosy2_decoder --local-dir models/cosy2_decoder

In [ ]:
!pip install hyperpyyaml pyyaml huggingface_hub onnxruntime

In [ ]:
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
# If you failed to clone the submodule due to network failures, please run the following command until success
%cd CosyVoice
!git submodule update --init --recursive

In [2]:
from huggingface_hub import snapshot_download

snapshot_download('FunAudioLLM/CosyVoice2-0.5B', local_dir='pretrained_models/CosyVoice2-0.5B')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

'/content/CosyVoice/pretrained_models/CosyVoice2-0.5B'

In [6]:
import os
import re
import sys
import uuid
from typing import List, Optional

import torch
import torchaudio
import torch.nn.functional as F
from hyperpyyaml import load_hyperpyyaml

project_root = "."
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, "third_party/Matcha-TTS"))

from cosyvoice.cli.frontend import CosyVoiceFrontEnd
from cosyvoice.utils.file_utils import load_wav


def extract_unit_ids_from_text(unit_text: str) -> List[int]:
    """
    "<123><45><6>" -> [123, 45, 6]
    """
    return list(map(int, re.findall(r"\d+", unit_text)))

def fade_in_out(fade_in_mel: torch.Tensor, fade_out_mel: torch.Tensor, window: np.ndarray) -> torch.Tensor:
    """
    Official LLaMA-Omni2 / CosyVoice-style overlap-add for streaming vocoder chunks.
    """
    device = fade_in_mel.device
    fade_in_mel = fade_in_mel.cpu()
    fade_out_mel = fade_out_mel.cpu()

    mel_overlap_len = int(window.shape[0] / 2)
    fade_in_mel[..., :mel_overlap_len] = (
        fade_in_mel[..., :mel_overlap_len] * window[:mel_overlap_len]
        + fade_out_mel[..., -mel_overlap_len:] * window[mel_overlap_len:]
    )
    return fade_in_mel.to(device)

class Cosy2Decoder:
    def __init__(
        self,
        model_dir: str = "models/cosy2_decoder",
        device: Optional[str] = None,
        hop_len: Optional[int] = None,
        load_jit: bool = False,
        load_trt: bool = False,
        load_onnx: bool = False,
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        with open(os.path.join(model_dir, "cosyvoice.yaml"), "r") as f:
            cfg = load_hyperpyyaml(
                f,
                overrides={
                    "qwen_pretrain_path": os.path.join(model_dir, "CosyVoice-BlankEN")
                },
            )

        self.frontend = CosyVoiceFrontEnd(
            cfg["get_tokenizer"],
            cfg["feat_extractor"],
            os.path.join(model_dir, "campplus.onnx"),
            os.path.join(model_dir, "speech_tokenizer_v2.onnx"),
            os.path.join(model_dir, "spk2info.pt"),
            False,
            cfg["allowed_special"],
        )
        self.sample_rate = cfg["sample_rate"]

        self.flow = cfg["flow"]
        self.flow.load_state_dict(
            torch.load(os.path.join(model_dir, "flow.pt"), map_location=self.device),
            strict=True,
        )
        self.flow.to(self.device).eval()
        self.flow.decoder.fp16 = False

        self.hift = cfg["hift"]
        hift_state_dict = {
            k.replace("generator.", ""): v
            for k, v in torch.load(os.path.join(model_dir, "hift.pt"), map_location=self.device).items()
        }
        self.hift.load_state_dict(hift_state_dict, strict=True)
        self.hift.to(self.device).eval()

        # Optional accelerators, same pattern as the official decoder.
        if load_jit:
            self.load_jit(os.path.join(model_dir, "flow.encoder.fp32.zip"))
        if load_trt and load_onnx:
            load_onnx = False
        if load_onnx:
            self.load_onnx(os.path.join(model_dir, "flow.decoder.estimator.fp32.onnx"))
        if load_trt:
            self.load_trt(os.path.join(model_dir, "flow.decoder.estimator.fp16.Volta.plan"))

        # IMPORTANT: these chunk-size settings are missing in the simplified notebook,
        # but they are part of the official decoder path.
        self.token_hop_len = hop_len if hop_len is not None else 2 * self.flow.input_frame_rate
        self.flow.encoder.static_chunk_size = 2 * self.flow.input_frame_rate
        self.flow.decoder.estimator.static_chunk_size = (
            2 * self.flow.input_frame_rate * self.flow.token_mel_ratio
        )

        # Streaming HiFT cache.
        self.mel_cache_len = 8
        self.source_cache_len = int(self.mel_cache_len * 480)
        self.speech_window = np.hamming(2 * self.source_cache_len)
        self.lock = threading.Lock()
        self.hift_cache_dict = {}

        for p in self.flow.parameters():
            p.requires_grad = False
        for p in self.hift.parameters():
            p.requires_grad = False

    def load_jit(self, flow_encoder_model: str):
        self.flow.encoder = torch.jit.load(flow_encoder_model, map_location=self.device)

    def load_onnx(self, flow_decoder_estimator_model: str):
        import onnxruntime

        option = onnxruntime.SessionOptions()
        option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
        option.intra_op_num_threads = 1
        providers = ["CUDAExecutionProvider" if torch.cuda.is_available() else "CPUExecutionProvider"]
        del self.flow.decoder.estimator
        self.flow.decoder.estimator = onnxruntime.InferenceSession(
            flow_decoder_estimator_model,
            sess_options=option,
            providers=providers,
        )

    def load_trt(self, flow_decoder_estimator_model: str):
        import tensorrt as trt

        del self.flow.decoder.estimator
        with open(flow_decoder_estimator_model, "rb") as f:
            self.flow.decoder.estimator_engine = trt.Runtime(
                trt.Logger(trt.Logger.INFO)
            ).deserialize_cuda_engine(f.read())
        self.flow.decoder.estimator = self.flow.decoder.estimator_engine.create_execution_context()
        self.flow.decoder.fp16 = True

    @torch.no_grad()
    def token2wav(
        self,
        token: torch.Tensor,
        prompt_token: torch.Tensor,
        prompt_feat: torch.Tensor,
        embedding: torch.Tensor,
        session_id: str,
        token_offset: int,
        finalize: bool = False,
        speed: float = 1.0,
    ) -> torch.Tensor:
        tts_mel, _ = self.flow.inference(
            token=token.to(self.device),
            token_len=torch.tensor([token.shape[1]], dtype=torch.int32, device=self.device),
            prompt_token=prompt_token.to(self.device),
            prompt_token_len=torch.tensor([prompt_token.shape[1]], dtype=torch.int32, device=self.device),
            prompt_feat=prompt_feat.to(self.device),
            prompt_feat_len=torch.tensor([prompt_feat.shape[1]], dtype=torch.int32, device=self.device),
            embedding=embedding.to(self.device),
            finalize=finalize,
        )

        # IMPORTANT: when decoding incrementally, crop away mel frames already emitted.
        tts_mel = tts_mel[:, :, token_offset * self.flow.token_mel_ratio:]

        if self.hift_cache_dict[session_id] is not None:
            hift_cache_mel = self.hift_cache_dict[session_id]["mel"]
            hift_cache_source = self.hift_cache_dict[session_id]["source"]
            tts_mel = torch.cat([hift_cache_mel, tts_mel], dim=2)
        else:
            hift_cache_source = torch.zeros(1, 1, 0, device=self.device)

        if not finalize:
            tts_speech, tts_source = self.hift.inference(
                speech_feat=tts_mel,
                cache_source=hift_cache_source,
            )
            if self.hift_cache_dict[session_id] is not None:
                tts_speech = fade_in_out(
                    tts_speech,
                    self.hift_cache_dict[session_id]["speech"],
                    self.speech_window,
                )
            self.hift_cache_dict[session_id] = {
                "mel": tts_mel[:, :, -self.mel_cache_len:],
                "source": tts_source[:, :, -self.source_cache_len:],
                "speech": tts_speech[:, -self.source_cache_len:],
            }
            tts_speech = tts_speech[:, :-self.source_cache_len]
        else:
            if speed != 1.0:
                assert self.hift_cache_dict[session_id] is None, "speed change only supports offline mode"
                tts_mel = F.interpolate(
                    tts_mel,
                    size=int(tts_mel.shape[2] / speed),
                    mode="linear",
                )
            tts_speech, _ = self.hift.inference(
                speech_feat=tts_mel,
                cache_source=hift_cache_source,
            )
            if self.hift_cache_dict[session_id] is not None:
                tts_speech = fade_in_out(
                    tts_speech,
                    self.hift_cache_dict[session_id]["speech"],
                    self.speech_window,
                )

        return tts_speech

    @torch.no_grad()
    def entry(
        self,
        generated_tokens: Union[List[int], torch.Tensor],
        prompt_speech_16k: torch.Tensor,
        stream: bool = False,
        speed: float = 1.0,
    ) -> torch.Tensor:
        if isinstance(generated_tokens, list):
            generated_tokens = torch.LongTensor(generated_tokens)
        generated_tokens = generated_tokens.to(self.device)
        if generated_tokens.ndim != 1:
            raise ValueError("generated_tokens must be shape (T,)")

        prompt_speech_feat = torch.zeros(1, 0, 80, device=self.device)
        prompt_speech_token = torch.zeros(1, 0, dtype=torch.int32, device=self.device)
        embedding = self.frontend._extract_spk_embedding(prompt_speech_16k).to(self.device)

        session_id = str(uuid.uuid1())
        with self.lock:
            self.hift_cache_dict[session_id] = None

        try:
            if stream:
                token_offset = 0
                outputs = []
                while True:
                    enough = len(generated_tokens) - token_offset >= self.token_hop_len + self.flow.pre_lookahead_len
                    if not enough:
                        break

                    cur_tokens = generated_tokens[: token_offset + self.token_hop_len + self.flow.pre_lookahead_len].unsqueeze(0)
                    cur_wav = self.token2wav(
                        token=cur_tokens,
                        prompt_token=prompt_speech_token,
                        prompt_feat=prompt_speech_feat,
                        embedding=embedding,
                        session_id=session_id,
                        token_offset=token_offset,
                        finalize=False,
                    )
                    token_offset += self.token_hop_len
                    outputs.append(cur_wav.cpu())

                final_tokens = generated_tokens.unsqueeze(0)
                final_wav = self.token2wav(
                    token=final_tokens,
                    prompt_token=prompt_speech_token,
                    prompt_feat=prompt_speech_feat,
                    embedding=embedding,
                    session_id=session_id,
                    token_offset=token_offset,
                    finalize=True,
                )
                outputs.append(final_wav.cpu())
                return torch.cat(outputs, dim=1)

            offline_tokens = generated_tokens.unsqueeze(0)
            wav = self.token2wav(
                token=offline_tokens,
                prompt_token=prompt_speech_token,
                prompt_feat=prompt_speech_feat,
                embedding=embedding,
                session_id=session_id,
                token_offset=0,
                finalize=True,
                speed=speed,
            )
            return wav.cpu()
        finally:
            with self.lock:
                self.hift_cache_dict.pop(session_id, None)

failed to import ttsfrd, use WeTextProcessing instead


ModuleNotFoundError: No module named 'tn'

In [ ]:
class Stage2SpeechSynthesizer:
    def __init__(
        self,
        decoder_dir: str = "models/cosy2_decoder",
        prompt_wav_path: Optional[str] = None,
        device: Optional[str] = None,
        lang: str = "en",
        hop_len: Optional[int] = None,
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.decoder = Cosy2Decoder(
            model_dir=decoder_dir,
            device=self.device,
            hop_len=hop_len,
        )

        if prompt_wav_path is None:
            prompt_wav_path = os.path.join(
                project_root,
                "llama_omni2/inference/prompt_zh.wav" if lang == "zh" else "llama_omni2/inference/prompt_en.wav",
            )
        self.prompt_speech_16k = load_wav(prompt_wav_path, 16000)

    @torch.no_grad()
    def units_text_to_waveform(
        self,
        unit_text: str,
        stream: bool = False,
        speed: float = 1.0,
    ):
        unit_ids = extract_unit_ids_from_text(unit_text)
        wav = self.decoder.entry(
            generated_tokens=unit_ids,
            prompt_speech_16k=self.prompt_speech_16k,
            stream=stream,
            speed=speed,
        )
        return wav, self.decoder.sample_rate

    @torch.no_grad()
    def units_to_waveform(
        self,
        unit_ids: Union[List[int], torch.Tensor],
        stream: bool = False,
        speed: float = 1.0,
    ):
        wav = self.decoder.entry(
            generated_tokens=unit_ids,
            prompt_speech_16k=self.prompt_speech_16k,
            stream=stream,
            speed=speed,
        )
        return wav, self.decoder.sample_rate

# Stage1(a) Train

data: <speech instruction, text response> pair

freeze: speech encoder

train: speech adapter + LLM (CE)


*batch size: 32 \
*3 epoch \
*peak learning rate: 5e-5

In [ ]:
from dataclasses import dataclass
from typing import Optional, List

from torch.nn.utils.rnn import pad_sequence
from transformers import get_cosine_schedule_with_warmup
import whisper


IGNORE_INDEX = -100
SPEECH_TOKEN_INDEX = -200
DEFAULT_SPEECH_TOKEN = "<speech>"


# -----------------------------
# 0. tokenizer helper
# -----------------------------
def ensure_speech_token(tokenizer, model=None, speech_token=DEFAULT_SPEECH_TOKEN):
    vocab = tokenizer.get_vocab()
    if speech_token not in vocab:
        tokenizer.add_special_tokens(
            {"additional_special_tokens": [speech_token]}
        )
        if model is not None:
            model.resize_token_embeddings(len(tokenizer))

    speech_token_id = tokenizer.convert_tokens_to_ids(speech_token)
    if speech_token_id is None:
        raise ValueError(f"failed to add/get speech token: {speech_token}")
    return speech_token_id


# -----------------------------
# 1. Whisper preprocessing
# -----------------------------
class WhisperMelBatchProcessor:
    """
    학습 코드 내부에서 raw wav path -> Whisper mel batch 생성
    출력:
        speech_tensor: (B, T, 128)
        speech_lengths: (B,)
    """

    def __init__(self, n_mels=128):
        self.n_mels = n_mels

    def load_one(self, path: str) -> torch.Tensor:
        speech = whisper.load_audio(path)
        speech = whisper.pad_or_trim(speech)
        speech = whisper.log_mel_spectrogram(speech, n_mels=self.n_mels).permute(1, 0)
        return speech  # (T, 128)

    def __call__(self, audio_paths: List[str]):
        feats = [self.load_one(p) for p in audio_paths]
        lengths = torch.tensor([x.size(0) for x in feats], dtype=torch.long)
        speech_tensor = pad_sequence(feats, batch_first=True, padding_value=0.0)
        return speech_tensor, lengths


# -----------------------------
# 2. Stage1(a) collator
# -----------------------------
class Stage1ACollator:
    """
    입력 sample 예시:
    {
        "audio_path": "...",
        "target_text": "...",
        ...
    }

    출력:
    {
        "input_ids": (B, L),
        "attention_mask": (B, L),
        "labels": (B, L),
        "audio_paths": list[str]
    }
    """

    def __init__(
        self,
        tokenizer,
        max_length=2048,
        speech_token=DEFAULT_SPEECH_TOKEN,
        ignore_index=IGNORE_INDEX,
    ):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.speech_token = speech_token
        self.ignore_index = ignore_index

        self.speech_token_id = tokenizer.convert_tokens_to_ids(speech_token)
        if self.speech_token_id is None:
            raise ValueError(
                f"{speech_token} not found in tokenizer. "
                "Run ensure_speech_token(tokenizer, model) first."
            )

    def _build_one(self, target_text: str):
        # prompt 부분: user가 speech를 준 상황
        prompt_messages = [
            {"role": "user", "content": self.speech_token}
        ]

        # full conversation
        full_messages = [
            {"role": "user", "content": self.speech_token},
            {"role": "assistant", "content": target_text},
        ]

        prompt_text = self.tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        full_text = self.tokenizer.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
            return_tensors=None,
        )["input_ids"]

        full_ids = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
            return_tensors=None,
        )["input_ids"]

        input_ids = torch.tensor(full_ids, dtype=torch.long)
        labels = input_ids.clone()

        # prompt 영역은 loss 제외
        prompt_len = min(len(prompt_ids), len(full_ids))
        labels[:prompt_len] = self.ignore_index

        # <speech> token -> speech placeholder index
        input_ids[input_ids == self.speech_token_id] = SPEECH_TOKEN_INDEX

        attention_mask = torch.ones_like(input_ids, dtype=torch.long)
        return input_ids, attention_mask, labels

    def __call__(self, batch):
        input_ids_list = []
        attention_mask_list = []
        labels_list = []
        audio_paths = []

        for sample in batch:
            target_text = sample["target_text"]
            audio_path = sample["audio_path"]

            input_ids, attention_mask, labels = self._build_one(target_text)

            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
            labels_list.append(labels)
            audio_paths.append(audio_path)

        input_ids = pad_sequence(
            input_ids_list,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        attention_mask = pad_sequence(
            attention_mask_list,
            batch_first=True,
            padding_value=0,
        )
        labels = pad_sequence(
            labels_list,
            batch_first=True,
            padding_value=self.ignore_index,
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "audio_paths": audio_paths,
        }


# -----------------------------
# 3. training config
# -----------------------------
@dataclass
class Stage1ATrainConfig:
    output_dir: str = "./stage1a_ckpt"
    num_epochs: int = 3
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    grad_accum_steps: int = 1
    max_grad_norm: float = 1.0
    log_every: int = 20
    save_every_epoch: bool = True
    num_workers: int = 2
    use_amp: bool = True


# -----------------------------
# 4. freeze policy
# -----------------------------
def freeze_for_stage1a(model):
    # 전체 trainable로 둔 뒤 speech encoder만 freeze
    for p in model.parameters():
        p.requires_grad = True

    speech_encoder = model.get_speech_encoder()
    if speech_encoder is not None:
        for p in speech_encoder.parameters():
            p.requires_grad = False
        speech_encoder.eval()


def get_trainable_params(model):
    return [p for p in model.parameters() if p.requires_grad]


# -----------------------------
# 5. checkpoint
# -----------------------------
def save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict() if optimizer is not None else None,
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": epoch,
            "step": step,
        },
        path,
    )
    tokenizer.save_pretrained(os.path.dirname(path))


# -----------------------------
# 6. eval
# -----------------------------
@torch.no_grad()
def evaluate_stage1a(
    model,
    data_loader,
    speech_batch_processor,
    device,
    amp_dtype,
):
    model.eval()

    total_loss = 0.0
    total_steps = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        speech_tensor, speech_lengths = speech_batch_processor(batch["audio_paths"])
        speech_tensor = speech_tensor.to(device)
        speech_lengths = speech_lengths.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
            enabled=(device.startswith("cuda"))
        ):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                speech=speech_tensor,
                speech_lengths=speech_lengths,
            )
            loss = outputs.loss

        total_loss += loss.item()
        total_steps += 1

    return total_loss / max(total_steps, 1)


# -----------------------------
# 7. train
# -----------------------------
def train_stage1a(
    model,
    tokenizer,
    train_dataset,
    val_dataset=None,
    batch_size=32,
    train_config=Stage1ATrainConfig(),
    device=None,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(train_config.output_dir, exist_ok=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    ensure_speech_token(tokenizer, model=model, speech_token=DEFAULT_SPEECH_TOKEN)

    collator = Stage1ACollator(
        tokenizer=tokenizer,
        max_length=getattr(model.config, "tokenizer_model_max_length", 2048),
        speech_token=DEFAULT_SPEECH_TOKEN,
        ignore_index=IGNORE_INDEX,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=train_config.num_workers,
        collate_fn=collator,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=train_config.num_workers,
            collate_fn=collator,
            pin_memory=torch.cuda.is_available(),
        )

    speech_batch_processor = WhisperMelBatchProcessor(n_mels=128)

    model.to(device)
    freeze_for_stage1a(model)

    optimizer = torch.optim.AdamW(
        get_trainable_params(model),
        lr=train_config.learning_rate,
        weight_decay=train_config.weight_decay,
    )

    total_update_steps = math.ceil(len(train_loader) / train_config.grad_accum_steps) * train_config.num_epochs
    warmup_steps = max(1, int(total_update_steps * train_config.warmup_ratio))

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    amp_dtype = (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(device.startswith("cuda") and amp_dtype == torch.float16 and train_config.use_amp)
    )

    global_step = 0
    best_val_loss = float("inf")

    print(f"device: {device}")
    print(f"train samples: {len(train_dataset)}")
    print(f"val samples: {len(val_dataset) if val_dataset is not None else 0}")
    print(f"batch size: {batch_size}")
    print(f"epochs: {train_config.num_epochs}")
    print(f"lr: {train_config.learning_rate}")
    print(f"warmup steps: {warmup_steps}")
    print(f"total update steps: {total_update_steps}")

    for epoch in range(train_config.num_epochs):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        accum_counter = 0

        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            speech_tensor, speech_lengths = speech_batch_processor(batch["audio_paths"])
            speech_tensor = speech_tensor.to(device, non_blocking=True)
            speech_lengths = speech_lengths.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype,
                enabled=(device.startswith("cuda") and train_config.use_amp)
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    speech=speech_tensor,
                    speech_lengths=speech_lengths,
                )
                loss = outputs.loss
                loss_for_backward = loss / train_config.grad_accum_steps

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            accum_counter += 1
            running_loss += loss.item()

            if accum_counter == train_config.grad_accum_steps:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        get_trainable_params(model),
                        train_config.max_grad_norm,
                    )
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(
                        get_trainable_params(model),
                        train_config.max_grad_norm,
                    )
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                accum_counter = 0
                global_step += 1

                if global_step % train_config.log_every == 0:
                    avg_loss = running_loss / (step + 1)
                    current_lr = scheduler.get_last_lr()[0]
                    print(
                        f"[Epoch {epoch+1}/{train_config.num_epochs}] "
                        f"step={global_step} "
                        f"loss={avg_loss:.4f} "
                        f"lr={current_lr:.7f}"
                    )

        # epoch 끝났는데 accumulation 잔여가 있으면 마지막 update
        if accum_counter > 0:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    get_trainable_params(model),
                    train_config.max_grad_norm,
                )
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(
                    get_trainable_params(model),
                    train_config.max_grad_norm,
                )
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        train_loss = running_loss / max(len(train_loader), 1)
        print(f"==> epoch {epoch+1} train_loss={train_loss:.4f}")

        if val_loader is not None:
            val_loss = evaluate_stage1a(
                model=model,
                data_loader=val_loader,
                speech_batch_processor=speech_batch_processor,
                device=device,
                amp_dtype=amp_dtype,
            )
            print(f"==> epoch {epoch+1} val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(
                    model=model,
                    tokenizer=tokenizer,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    step=global_step,
                    path=os.path.join(train_config.output_dir, "best_stage1a.pt"),
                )
                print(f"saved best checkpoint: val_loss={val_loss:.4f}")

        if train_config.save_every_epoch:
            save_checkpoint(
                model=model,
                tokenizer=tokenizer,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                step=global_step,
                path=os.path.join(train_config.output_dir, f"epoch_{epoch+1}.pt"),
            )

In [ ]:
cfg = Stage1ATrainConfig(
    output_dir="./stage1a_ckpt",
    num_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    grad_accum_steps=1,
    log_every=20,
    save_every_epoch=True,
)

train_stage1a(
    model=model,
    tokenizer=tokenizer,
    train_dataset=stage1a_dataset,
    batch_size=32,
    train_config=cfg,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

device: cuda
train samples: 26249
val samples: 0
batch size: 32
epochs: 3
lr: 5e-05
warmup steps: 73
total update steps: 2463


/tmp/ipykernel_10966/3462986066.py:368: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[Epoch 1/3] step=20 loss=10.9353 lr=0.0000137
[Epoch 1/3] step=40 loss=10.4290 lr=0.0000274


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.71 GiB. GPU 0 has a total capacity of 79.25 GiB of which 1.17 GiB is free. Including non-PyTorch memory, this process has 78.07 GiB memory in use. Of the allocated memory 61.22 GiB is allocated by PyTorch, and 16.34 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Stage1(b) Train

In [14]:
import os
import re
import math
from dataclasses import dataclass
from typing import List, Optional

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import get_cosine_schedule_with_warmup


IGNORE_INDEX = -100

class Stage1BCollator:
    """
    LLMSpeechGenerator.forward_stage1b(...) 용 batch 생성

    반환:
    {
        "text_token_ids": (B, T_text)
        "text_attention_mask": (B, T_text)
        "unit_token_ids": (B, T_unit)
        "unit_attention_mask": (B, T_unit)
    }
    """

    def __init__(
        self,
        tts_tokenizer,
        max_text_len: Optional[int] = None,
        max_unit_len: Optional[int] = None,
    ):
        self.tokenizer = tts_tokenizer
        self.max_text_len = max_text_len
        self.max_unit_len = max_unit_len

        if self.tokenizer.pad_token_id is None:
            if self.tokenizer.eos_token_id is not None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            else:
                raise ValueError("tts_tokenizer must have pad_token_id or eos_token_id")

    @staticmethod
    def parse_unit_tokens(unit_str: str) -> List[str]:
        # "<1252><1611><1117>" -> ["<1252>", "<1611>", "<1117>"]
        return re.findall(r"<\d+>", unit_str)

    def encode_text(self, text: str) -> torch.LongTensor:
        ids = self.tokenizer(
            text,
            add_special_tokens=False,
            truncation=(self.max_text_len is not None),
            max_length=self.max_text_len,
            return_tensors=None,
        )["input_ids"]

        if len(ids) == 0:
            raise ValueError(f"text tokenization produced empty ids: {text[:80]}")
        return torch.tensor(ids, dtype=torch.long)

    def encode_units(self, unit_str: str) -> torch.LongTensor:
        unit_tokens = self.parse_unit_tokens(unit_str)
        if len(unit_tokens) == 0:
            raise ValueError(f"unit string parsing failed: {unit_str[:80]}")

        ids = []
        unk_id = getattr(self.tokenizer, "unk_token_id", None)

        for tok in unit_tokens:
            tok_id = self.tokenizer.convert_tokens_to_ids(tok)
            if tok_id is None or (unk_id is not None and tok_id == unk_id):
                raise ValueError(
                    f"unit token {tok} not found in tts tokenizer vocab. "
                    "Do not pass raw unit integers directly; map through tts tokenizer."
                )
            ids.append(tok_id)

        if self.max_unit_len is not None:
            ids = ids[:self.max_unit_len]

        if len(ids) == 0:
            raise ValueError("unit ids became empty after truncation")

        return torch.tensor(ids, dtype=torch.long)

    def __call__(self, batch):
        text_ids_list = []
        unit_ids_list = []

        for sample in batch:
            text_ids = self.encode_text(sample["text"])
            unit_ids = self.encode_units(sample["unit_str"])

            text_ids_list.append(text_ids)
            unit_ids_list.append(unit_ids)

        text_token_ids = pad_sequence(
            text_ids_list,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        unit_token_ids = pad_sequence(
            unit_ids_list,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )

        text_attention_mask = (text_token_ids != self.tokenizer.pad_token_id).long()
        unit_attention_mask = (unit_token_ids != self.tokenizer.pad_token_id).long()

        return {
            "text_token_ids": text_token_ids,
            "text_attention_mask": text_attention_mask,
            "unit_token_ids": unit_token_ids,
            "unit_attention_mask": unit_attention_mask,
        }

# Train config
@dataclass
class Stage1BTrainConfig:
    output_dir: str = "./stage1b_ckpt"
    num_epochs: int = 5
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    grad_accum_steps: int = 1
    max_grad_norm: float = 1.0
    log_every: int = 20
    save_every_epoch: bool = True
    num_workers: int = 2
    use_amp: bool = True


# Freeze policy
def freeze_for_stage1b(tts_model):
    """
    - gate fusion bypass
    - Qwen2 TTS LM만 학습
    """
    if hasattr(tts_model, "set_training_stage"):
        tts_model.set_training_stage("stage1b")
        return

    # fallback
    for p in tts_model.parameters():
        p.requires_grad = True

    if hasattr(tts_model, "input_proj"):
        for p in tts_model.input_proj.parameters():
            p.requires_grad = False

    if hasattr(tts_model, "gate_fusion"):
        for p in tts_model.gate_fusion.parameters():
            p.requires_grad = False

    if hasattr(tts_model, "gate"):
        for p in tts_model.gate.parameters():
            p.requires_grad = False


def get_trainable_params(module):
    return [p for p in module.parameters() if p.requires_grad]


# Checkpoint
def save_stage1b_checkpoint(tts_model, optimizer, scheduler, epoch, step, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    ckpt = {
        "model": tts_model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "step": step,
    }
    torch.save(ckpt, path)

    if hasattr(tts_model, "tokenizer") and tts_model.tokenizer is not None:
        tts_model.tokenizer.save_pretrained(os.path.dirname(path))


# Eval
@torch.no_grad()
def evaluate_stage1b(
    tts_model,
    data_loader,
    device,
    amp_dtype,
    use_amp=True,
):
    tts_model.eval()
    device_type = "cuda" if str(device).startswith("cuda") else "cpu"

    total_loss = 0.0
    total_steps = 0

    for batch in data_loader:
        text_token_ids = batch["text_token_ids"].to(device, non_blocking=True)
        text_attention_mask = batch["text_attention_mask"].to(device, non_blocking=True)
        unit_token_ids = batch["unit_token_ids"].to(device, non_blocking=True)
        unit_attention_mask = batch["unit_attention_mask"].to(device, non_blocking=True)

        with torch.autocast(
            device_type=device_type,
            dtype=amp_dtype,
            enabled=(device_type == "cuda" and use_amp),
        ):
            outputs = tts_model.forward_stage1b(
                text_token_ids=text_token_ids,
                unit_token_ids=unit_token_ids,
                text_attention_mask=text_attention_mask,
                unit_attention_mask=unit_attention_mask,
                append_sep=True,
                return_dict=True,
            )
            loss = outputs.loss

        total_loss += loss.item()
        total_steps += 1

    return total_loss / max(total_steps, 1)


# Train
def train_stage1b(
    tts_model,
    tts_tokenizer,
    train_dataset,
    batch_size=32,
    train_config = None,
    device=None,
    max_text_len= None,
    max_unit_len= None,
):
    if train_config is None:
        train_config = Stage1BTrainConfig()

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(train_config.output_dir, exist_ok=True)

    if tts_tokenizer.pad_token is None:
        if tts_tokenizer.eos_token is not None:
            tts_tokenizer.pad_token = tts_tokenizer.eos_token
        else:
            raise ValueError("tts_tokenizer must have pad_token or eos_token")

    collator = Stage1BCollator(
        tts_tokenizer=tts_tokenizer,
        max_text_len=max_text_len,
        max_unit_len=max_unit_len,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=train_config.num_workers,
        collate_fn=collator,
        pin_memory=torch.cuda.is_available(),
    )

    tts_model.to(device)
    freeze_for_stage1b(tts_model)

    # training 때 cache 끄기
    if hasattr(tts_model, "model") and hasattr(tts_model.model.config, "use_cache"):
        tts_model.model.config.use_cache = False

    optimizer = torch.optim.AdamW(
        get_trainable_params(tts_model),
        lr=train_config.learning_rate,
        weight_decay=train_config.weight_decay,
    )

    total_update_steps = math.ceil(len(train_loader) / train_config.grad_accum_steps) * train_config.num_epochs
    warmup_steps = max(1, int(total_update_steps * train_config.warmup_ratio))

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    device_type = "cuda" if str(device).startswith("cuda") else "cpu"
    amp_dtype = (
        torch.bfloat16
        if device_type == "cuda" and torch.cuda.is_bf16_supported()
        else torch.float16
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device_type == "cuda" and amp_dtype == torch.float16 and train_config.use_amp),
    )

    global_step = 0
    best_val_loss = float("inf")

    print(f"device: {device}")
    print(f"train samples: {len(train_dataset)}")
    print(f"batch size: {batch_size}")
    print(f"epochs: {train_config.num_epochs}")
    print(f"lr: {train_config.learning_rate}")
    print(f"warmup steps: {warmup_steps}")
    print(f"total update steps: {total_update_steps}")

    for epoch in range(train_config.num_epochs):
        tts_model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        accum_counter = 0

        for step, batch in enumerate(train_loader):
            text_token_ids = batch["text_token_ids"].to(device, non_blocking=True)
            text_attention_mask = batch["text_attention_mask"].to(device, non_blocking=True)
            unit_token_ids = batch["unit_token_ids"].to(device, non_blocking=True)
            unit_attention_mask = batch["unit_attention_mask"].to(device, non_blocking=True)

            with torch.autocast(
                device_type=device_type,
                dtype=amp_dtype,
                enabled=(device_type == "cuda" and train_config.use_amp),
            ):
                outputs = tts_model.forward_stage1b(
                    text_token_ids=text_token_ids,
                    unit_token_ids=unit_token_ids,
                    text_attention_mask=text_attention_mask,
                    unit_attention_mask=unit_attention_mask,
                    append_sep=True,
                    return_dict=True,
                )
                loss = outputs.loss
                loss_for_backward = loss / train_config.grad_accum_steps

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            accum_counter += 1
            running_loss += loss.item()

            if accum_counter == train_config.grad_accum_steps:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        get_trainable_params(tts_model),
                        train_config.max_grad_norm,
                    )
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(
                        get_trainable_params(tts_model),
                        train_config.max_grad_norm,
                    )
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                accum_counter = 0
                global_step += 1

                if global_step % train_config.log_every == 0:
                    avg_loss = running_loss / (step + 1)
                    current_lr = scheduler.get_last_lr()[0]
                    print(
                        f"[Epoch {epoch+1}/{train_config.num_epochs}] "
                        f"step={global_step} "
                        f"loss={avg_loss:.4f} "
                        f"lr={current_lr:.7f}"
                    )

        if accum_counter > 0:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    get_trainable_params(tts_model),
                    train_config.max_grad_norm,
                )
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(
                    get_trainable_params(tts_model),
                    train_config.max_grad_norm,
                )
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        train_loss = running_loss / max(len(train_loader), 1)
        print(f"==> epoch {epoch+1} train_loss={train_loss:.4f}")

        if train_config.save_every_epoch:
            save_stage1b_checkpoint(
                tts_model=tts_model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                step=global_step,
                path=os.path.join(train_config.output_dir, f"epoch_{epoch+1}.pt"),
            )

In [15]:
from types import SimpleNamespace

config = SimpleNamespace(
    hidden_size=896,
    speech_generator={
        "architectures": ["Qwen2ForCausalLM"],
        "attention_dropout": 0.0,
        "bos_token_id": 151643,
        "eos_token_id": 151643,
        "hidden_act": "silu",
        "hidden_size": 896,
        "initializer_range": 0.02,
        "intermediate_size": 4864,
        "max_position_embeddings": 32768,
        "max_window_layers": 24,
        "model_type": "qwen2",
        "num_attention_heads": 14,
        "num_hidden_layers": 24,
        "num_key_value_heads": 2,
        "rms_norm_eps": 1e-6,
        "rope_theta": 1000000.0,
        "sliding_window": None,
        "tie_word_embeddings": True,
        "torch_dtype": "bfloat16",
        "transformers_version": "4.43.4",
        "use_cache": True,
        "use_mrope": False,
        "use_sliding_window": False,
        "vocab_size": 158227,
    },
    stream_params="(3,10)",
    tokenizer_model_max_length=4096,
    unit_vocab_size=6561,
)

In [16]:
import json
from types import SimpleNamespace
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

tts_tokenizer = AutoTokenizer.from_pretrained(
    "ICTNLP/LLaMA-Omni2-0.5B",
    subfolder="tts_tokenizer",
    trust_remote_code=True,
)

if tts_tokenizer.pad_token is None:
    tts_tokenizer.pad_token = tts_tokenizer.eos_token

tts_model = LLMSpeechGenerator(config, tts_tokenizer)

#train config
cfg = Stage1BTrainConfig(
    output_dir="./stage1b_ckpt",
    num_epochs=5,
    learning_rate=5e-4,
    warmup_ratio=0.03,
    grad_accum_steps=1,
    log_every=20,
    save_every_epoch=True,
)

#train
train_stage1b(
    tts_model=tts_model,
    tts_tokenizer=tts_tokenizer,
    train_dataset=stage1b_dataset,
    batch_size=4,
    train_config=cfg,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

device: cuda
train samples: 26251
batch size: 4
epochs: 5
lr: 0.0005
warmup steps: 984
total update steps: 32815
[Epoch 1/5] step=20 loss=12.0922 lr=0.0000102
[Epoch 1/5] step=40 loss=11.8220 lr=0.0000203
[Epoch 1/5] step=60 loss=11.3902 lr=0.0000305
[Epoch 1/5] step=80 loss=10.8751 lr=0.0000407
[Epoch 1/5] step=100 loss=10.2703 lr=0.0000508
[Epoch 1/5] step=120 loss=9.6610 lr=0.0000610
[Epoch 1/5] step=140 loss=9.1484 lr=0.0000711
[Epoch 1/5] step=160 loss=8.7424 lr=0.0000813
[Epoch 1/5] step=180 loss=8.4183 lr=0.0000915
[Epoch 1/5] step=200 loss=8.1527 lr=0.0001016
[Epoch 1/5] step=220 loss=7.9251 lr=0.0001118
[Epoch 1/5] step=240 loss=7.7294 lr=0.0001220
[Epoch 1/5] step=260 loss=7.5555 lr=0.0001321
[Epoch 1/5] step=280 loss=7.3964 lr=0.0001423
[Epoch 1/5] step=300 loss=7.2530 lr=0.0001524
[Epoch 1/5] step=320 loss=7.1222 lr=0.0001626
[Epoch 1/5] step=340 loss=7.0036 lr=0.0001728
[Epoch 1/5] step=360 loss=6.8909 lr=0.0001829
[Epoch 1/5] step=380 loss=6.7879 lr=0.0001931
[Epoch 1/5] 

# Stage2 Train

In [ ]:
tts_model.set_training_stage("stage2")

out = tts_model.forward_stage2(
    text_token_ids=text_ids_tts,
    unit_token_ids=unit_ids_tts,
    llm_hidden_states=assistant_hidden_states,   # (B, N, 896)
    text_attention_mask=text_mask_tts,
    unit_attention_mask=unit_mask_tts,
)
loss = out.loss